#Working with Numbers, Strings, Dates, and Nulls

In [0]:
#creates the dataframe.
df = spark.read.table("samples.bakehouse.sales_transactions")

#Display the data frame.
display(df)

In [0]:
#Code to seek out records that fit the request.
#df2=df.where("paymentMethod == 'mastercard'")

#Code to seek out records that do not fit the request
df2=df.where("paymentMethod != 'mastercard'")

#Display the dataframe.
display(df2)

In [0]:
%sql
--SELECT * FROM Samples.bakehouse.sales_transactions where paymentMethod ='mastercard' --Code to seek out records that fit the request.
SELECT * FROM Samples.bakehouse.sales_transactions where paymentMethod !='mastercard' --Code to seek out records that fit the request.

In [0]:
#Import code
from pyspark.sql.functions import expr, pow, round

#Creates a new column with quanity to the second power then dividing by 6
#df2=df.withColumn("myCol",pow("quantity",2)/6)

#Creates a new column with quanity to the second power, divides by 6 then rounds it to 2 (decimal)
df2=df.withColumn("myCol",round(pow("quantity",2)/6,2))

#Display the dataframe.
display(df2)

In [0]:
%sql
SELECT *, ROUND(pow(quantity,2)/6,2) FROM samples.bakehouse.sales_transactions --Creates a new column with quanity to the second power, divides by 6 then rounds it to 2 (decimal)

In [0]:
#selects the product column and does the following: lower product makes the text lowercase. Upper product makes the text upper case/all caps. Trim product removes leading and trailing spaces. Substr product takes the first 4 characters of the product. Split product splits the product into an array based on the space character.
df2=df.select("product").\
    withColumn("lower_product",expr("lower(product)")).\
    withColumn("upper_product",expr("upper(product)")).\
    withColumn("trim_product",expr("trim(product)")).\
    withColumn("substr_product",expr("substr(product,0,4)")).\
    withColumn("split_product",expr("split(product,' ')"))

#Display the dataframe.
display(df2)

In [0]:
%sql
--selects the product column and does the following: lower product makes the text lowercase. Upper product makes the text upper case/all caps. Trim product removes leading and trailing spaces. Substr product takes the first 4 characters of the product. Split product splits the product into an array based on the space character.
SELECT
  product,
  lower(product) AS lower_product,
  upper(product) AS upper_product,
  trim(product) AS trim_product,
  substr(product, 0, 4) AS substr_product,
  split(product, ' ') AS split_product
FROM samples.bakehouse.sales_transactions

In [0]:
#Import code
from pyspark.sql.functions import regexp_extract, regexp_replace

#First line of code extracts the middle word of the product column. Second line of code replaces the string with AAAA.
df2=df.select("product").\
    withColumn("regexp_extract_product",expr("regexp_extract(product, r'^\S+\s+(\S+)\s+\S+$', 1)")).\
    withColumn("regexp_replace_product",expr("regexp_replace(product, r'^\S+\s+(\S+)\s+\S+$', 'AAAA')"))

#Display the dataframe.
display(df2)

In [0]:
%sql
--#First line of code extracts the middle word of the product column. Second line of code replaces the string with AAAA.
SELECT
  product,
  regexp_extract(product, r'^\S+\s+(\S+)\s+\S+$', 1) AS regexp_extract_product,
  regexp_replace(product, r'^\S+\s+(\S+)\s+\S+$', 'AAAA') AS regexp_replace_product
FROM samples.bakehouse.sales_transactions

In [0]:
#Import code
from pyspark.sql.functions import current_date, current_timestamp, to_date,date_add,date_sub,lit,to_timestamp

#Creates specific columns for dates, time, and custom date and timestamps.
df2=df.select("dateTime")\
    .withColumn("today", current_date())\
    .withColumn("now", current_timestamp())\
    .withColumn("dateTime_date", to_date("dateTime"))\
    .withColumn("dateTime_date_add", date_add(to_date("dateTime"), 5))\
    .withColumn("dateTime_date_sub", date_sub(to_date("dateTime"), 5))\
    .withColumn("custom_date", to_date(lit("20251014"), 'yyyyMMdd'))\
    .withColumn("custom_timestamp", to_timestamp(lit("20251014112233"),'yyyyMMddHHmmss'))
#Display the dataframe.
display(df2)

In [0]:
%sql
SELECT
  dateTime,
  current_date() AS today,
  current_timestamp() AS now,
  to_date(dateTime) AS dateTime_date,
  date_add(to_date(dateTime), 5) AS dateTime_date_add,
  date_sub(to_date(dateTime), 5) AS dateTime_date_sub,
  to_date('20251014', 'yyyyMMdd') AS custom_date,
  to_timestamp('20251014112233', 'yyyyMMddHHmmss') AS custom_timestamp
FROM samples.bakehouse.sales_transactions

In [0]:
#Import code
from pyspark.sql.functions import coalesce, expr

#Function to look at columns and see if there are null values, take them, and then move on to the next column.
df2 = df.select("transactionID", "customerID")\
    .withColumn("coalesce_id", coalesce("transactionID", "customerID"))\
    .withColumn("if_else", expr("CASE WHEN transactionID IS NOT NULL THEN transactionID ELSE customerID END"))\
    .na.drop(subset=["transactionID"]) #specifies a column
    #.na.drop('all') #drops all columns
    #.na.fill('myvalue') #fills all columns with a myvalue

#Display the dataframe.
display(df2)

In [0]:
%sql

--Function to look at columns and see if there are null values, take them, and then move on to the next column.
SELECT
  transactionID,
  customerID,
  coalesce(transactionID, customerID) AS coalesce_id,
  CASE WHEN transactionID IS NOT NULL THEN transactionID ELSE customerID END AS if_else
  FROM samples.bakehouse.sales_transactions